In [1]:
import numpy as np
import scipy.integrate as integrate
import sympy as sp
import math

In [2]:
# Aufgabenblatt 3, Aufgabe 2.3

# Intervall [-pi, pi] und Ziel-Funktion
a, b = -np.pi, np.pi
f = lambda t: np.cos(t)

# --- 1. Unterraum V1: Monome phi_k(t) = t^k ---
G1 = np.zeros((6, 6))
b1 = np.zeros(6)

for i in range(6):
    b1[i], _ = integrate.quad(lambda t: (t**i) * f(t), a, b)
    for j in range(6):
        G1[i, j], _ = integrate.quad(lambda t: (t**i) * (t**j), a, b)

c1 = np.linalg.solve(G1, b1)
p1 = lambda t: sum(c1[k] * (t**k) for k in range(6))
err1 = np.sqrt(integrate.quad(lambda t: (f(t) - p1(t))**2, a, b)[0])

# --- 2. Unterraum V2: Gauß-Funktionen ---
t_nodes = np.array([-np.pi + k * (2 * np.pi / 5.0) for k in range(6)])
phi_v2 = lambda k, t: np.exp(-(t - t_nodes[k])**2)

G2 = np.zeros((6, 6))
b2 = np.zeros(6)

for i in range(6):
    b2[i], _ = integrate.quad(lambda t: phi_v2(i, t) * f(t), a, b)
    for j in range(6):
        G2[i, j], _ = integrate.quad(lambda t: phi_v2(i, t) * phi_v2(j, t), a, b)

c2 = np.linalg.solve(G2, b2)
p2 = lambda t: sum(c2[k] * phi_v2(k, t) for k in range(6))
err2 = np.sqrt(integrate.quad(lambda t: (f(t) - p2(t))**2, a, b)[0])

# --- Ausgabe ---
print(f"L2-Fehler V1:  {err1:.5f}")
print(f"L2-Fehler V2:  {err2:.5f}")

L2-Fehler V1:  0.04602
L2-Fehler V2:  0.03572


In [3]:
# Aufgabenblatt 4, Aufgabe 2.1

x = sp.Symbol('x')

# 1. Rekursionsformel: H_{n+1}(x) = 2x*H_n(x) - 2n*H_{n-1}(x)
H_rec = [sp.Integer(1)]  # H_0 = 1
for n in range(5):
    h_prev = H_rec[-2] if n > 0 else sp.Integer(0)
    h_next = sp.expand(2 * x * H_rec[-1] - 2 * n * h_prev)
    H_rec.append(h_next)

# 2. Ableitung (Rodrigues-Formel): H_n(x) = (-1)^n * exp(x^2) * d^n/dx^n (exp(-x^2))
H_rodr = []
for n in range(6):
    dn = sp.diff(sp.exp(-x**2), (x, n))
    val = sp.expand((-1)**n * sp.exp(x**2) * dn)
    H_rodr.append(val)

# 3. Potenzreihe / Summenformel mit k1 + 2*k2 = n
# (Hinweis: Im PDF steht (-1)^(k1+k2); die Standardformel lautet (-1)^k2)
H_sum = []
for n in range(6):
    s = 0
    for k2 in range(n // 2 + 1):
        k1 = n - 2 * k2
        coeff = (
            math.factorial(n) // (math.factorial(k1) * math.factorial(k2))
        ) * ((-1)**k2)
        s += coeff * (2 * x) ** k1
    H_sum.append(sp.expand(s))

# Ausgabe als Tabelle
print(f"{'n':<3} | {'Rekursionsformel':<30} | {'Ableitung (Rodrigues)':<30} | {'Potenzreihe'}")
print("-" * 95)
for i in range(6):
    print(f"{i:<3} | {str(H_rec[i]):<30} | {str(H_rodr[i]):<30} | {H_sum[i]}")

n   | Rekursionsformel               | Ableitung (Rodrigues)          | Potenzreihe
-----------------------------------------------------------------------------------------------
0   | 1                              | 1                              | 1
1   | 2*x                            | 2*x                            | 2*x
2   | 4*x**2 - 2                     | 4*x**2 - 2                     | 4*x**2 - 2
3   | 8*x**3 - 12*x                  | 8*x**3 - 12*x                  | 8*x**3 - 12*x
4   | 16*x**4 - 48*x**2 + 12         | 16*x**4 - 48*x**2 + 12         | 16*x**4 - 48*x**2 + 12
5   | 32*x**5 - 160*x**3 + 120*x     | 32*x**5 - 160*x**3 + 120*x     | 32*x**5 - 160*x**3 + 120*x
